### description :Estimate LD span for SNPs based on LD-decay within genomic windows
### author :RoshanAbeyratne & HariChhetri
### date :20022025

In [23]:
rm(list=ls())
library(tidyverse)

# change these variables as needed
wdir = "/lustre/orion/syb111/proj-shared/Personal/abeyratnecr/pennycress/sub_proj/snp2genes_growout1/github_sync"
filtered_snps_dir = paste0(wdir, "/fsnps/filtered_snps")
trait_list_fn = paste0(wdir, '/fsnps/gwas_trait_list.txt')
window_list_fn = paste0(wdir, "/pw_ld/5000000_2500000.bed") # filename of the bed file with genomic intervals for LD estimation

setwd(wdir)

In [24]:
trait_list <- read.table(trait_list_fn, header=F) %>% as.matrix() %>% as.vector()
length(trait_list)
trait_list

[1] 78

[1] "tgw"                           "rr_area"                      
 [3] "area"                          "circularity"                  
 [5] "color_RGB_green_mean"          "perimeter_crofton"            
 [7] "axis_major_length"             "color_LAB_A_mean"             
 [9] "color_LAB_B_mean"              "color_LAB_L_mean"             
[11] "color_LAB_B_std"               "feret_diameter_max"           
[13] "extent"                        "rr_height"                    
[15] "color_RGB_blue_std"            "color_LAB_L_std"              
[17] "area_convex"                   "axis_minor_length"            
[19] "rr_hypotenuse"                 "color_RGB_blue_mean"          
[21] "yield_kg"                      "aspect_ratio"                 
[23] "color_RGB_red_std"             "perimeter"                    
[25] "color_HSB_saturation_std"      "color_HSB_hue_mean"           
[27] "roundness"                     "color_RGB_green_std"          
[29] "eccentricity"                  "color_HSB_hue_std"            
[31] "tillers"                       "equivalent_diameter_area"     
[33] "color_HSB_brightness_std"      "color_HSB_saturation_mean"    
[35] "solidity"                      "color_RGB_red_mean"           
[37] "color_LAB_A_std"               "rr_width"                     
[39] "color_HSB_brightness_mean"     "variety"                      
[41] "oil_percent_dwb"               "days_to_50_percent_flower"    
[43] "stem_branch"                   "c16_0_methyl_palmitate"       
[45] "seed_length"                   "f3"                           
[47] "days_to_maturity"              "days_to_1st_flower"           
[49] "floral_area_plant"             "pod_black_spots"              
[51] "first_pod_height"              "seed_width"                   
[53] "avg_rosette"                   "c18_3_methyl_linolenate"      
[55] "plot_rating_at_maturity"       "floral_branch"                
[57] "lw_ratio"                      "c22_1_methyl_erucate"         
[59] "f2"                            "sinigrin_umol_g"              
[61] "stem_lesions"                  "days_to_100_percent_flower"   
[63] "lodging"                       "plant_height"                 
[65] "seed_area"                     "c18_2_methyl_linoleate"       
[67] "c20_1_methyl_eicosenoate"      "f0"                           
[69] "moisture"                      "c18_1_methyl_oleate"          
[71] "f1"                            "f4"                           
[73] "other"                         "days_to_50_percent_emergence" 
[75] "days_to_budding"               "c22_0_methyl_behenate"        
[77] "f5"                            "days_to_100_percent_emergence"

In [25]:
# Combine all GWAS hits passing p-value threshold
models <- c("BLINK", "FarmCPU", "MLMM", "MLM")
gwas_master <- NULL
for (i in 1:length(trait_list)) {
    for (j in 1:length(models)) {
        tmp_df <- read.csv(paste0(filtered_snps_dir, '/GAPIT.Association.GWAS_Results.',models[j], '.', trait_list[i], '_filtered.csv'))
        if (nrow(tmp_df) == 0) {
            stop('error no lines in file!')
        } else {
            gwas_master <- rbind(gwas_master, cbind(tmp_df,
                                                   trait=trait_list[i],
                                                   model=models[j]))
        }
    }
}

# Since a row of NAs was assigned for each trait-model combination if no hits were present that passed a user defined threshold, I am removing them here.
gwas_master_ss <- gwas_master %>% filter(!is.na(SNP))
# dim(gwas_master)
# dim(gwas_master_ss)

cat(paste0('Number of original GWAS hits: ', nrow(gwas_master), "\n"))
cat(paste0('Number of filtered GWAS hits: ', nrow(gwas_master_ss), "\n"))

str(gwas_master_ss)

Number of original GWAS hits: 2898
Number of filtered GWAS hits: 2700
'data.frame':	2700 obs. of  10 variables:
 $ SNP        : chr  "2_68565136" "2_68585251" "3_5499477" "3_27542756" ...
 $ Chr        : int  2 2 3 3 2 3 2 2 2 2 ...
 $ Pos        : int  68565136 68585251 5499477 27542756 68585251 5499477 68585251 68506350 68506364 68506412 ...
 $ P.value    : num  1.08e-10 1.62e-08 4.45e-10 5.23e-09 2.44e-12 ...
 $ MAF        : num  0.0031 0.2229 0.0557 0.0341 0.2229 ...
 $ nobs       : int  323 323 323 323 323 323 323 323 323 323 ...
 $ H.B.P.Value: num  2.14e-04 7.98e-03 4.40e-04 3.44e-03 4.83e-06 ...
 $ Effect     : num  0.2685 0.0246 0.0447 -0.0542 0.0272 ...
 $ trait      : chr  "tgw" "tgw" "tgw" "tgw" ...
 $ model      : chr  "BLINK" "BLINK" "BLINK" "BLINK" ...


In [26]:
# What are the traits with at least one significant GWAS hit across all GAPIT models?
traits_w_sig_hits <- gwas_master_ss %>% pull(trait) %>% unique() %>% sort()
length(traits_w_sig_hits)
traits_w_sig_hits

[1] 38

[1] "area_convex"                   "aspect_ratio"                 
 [3] "axis_minor_length"             "c16_0_methyl_palmitate"       
 [5] "c18_1_methyl_oleate"           "c18_2_methyl_linoleate"       
 [7] "c18_3_methyl_linolenate"       "c20_1_methyl_eicosenoate"     
 [9] "c22_0_methyl_behenate"         "c22_1_methyl_erucate"         
[11] "days_to_100_percent_emergence" "days_to_100_percent_flower"   
[13] "days_to_1st_flower"            "days_to_50_percent_emergence" 
[15] "days_to_budding"               "eccentricity"                 
[17] "extent"                        "f0"                           
[19] "f1"                            "f2"                           
[21] "f3"                            "f4"                           
[23] "f5"                            "first_pod_height"             
[25] "lodging"                       "lw_ratio"                     
[27] "oil_percent_dwb"               "other"                        
[29] "plant_height"                  "plot_rating_at_maturity"      
[31] "roundness"                     "rr_width"                     
[33] "seed_area"                     "seed_length"                  
[35] "seed_width"                    "tgw"                          
[37] "variety"                       "yield_kg"

In [27]:
# What traits dont have any GWAS hits across all GAPIT models?
traits_no_hits <- gwas_master %>% filter(!(trait %in% traits_w_sig_hits)) %>% pull(trait) %>% unique() %>% sort()
length(traits_no_hits)
traits_no_hits

[1] 40

[1] "area"                      "avg_rosette"              
 [3] "axis_major_length"         "circularity"              
 [5] "color_HSB_brightness_mean" "color_HSB_brightness_std" 
 [7] "color_HSB_hue_mean"        "color_HSB_hue_std"        
 [9] "color_HSB_saturation_mean" "color_HSB_saturation_std" 
[11] "color_LAB_A_mean"          "color_LAB_A_std"          
[13] "color_LAB_B_mean"          "color_LAB_B_std"          
[15] "color_LAB_L_mean"          "color_LAB_L_std"          
[17] "color_RGB_blue_mean"       "color_RGB_blue_std"       
[19] "color_RGB_green_mean"      "color_RGB_green_std"      
[21] "color_RGB_red_mean"        "color_RGB_red_std"        
[23] "days_to_50_percent_flower" "days_to_maturity"         
[25] "equivalent_diameter_area"  "feret_diameter_max"       
[27] "floral_area_plant"         "floral_branch"            
[29] "moisture"                  "perimeter"                
[31] "perimeter_crofton"         "pod_black_spots"          
[33] "rr_area"                   "rr_height"                
[35] "rr_hypotenuse"             "sinigrin_umol_g"          
[37] "solidity"                  "stem_branch"              
[39] "stem_lesions"              "tillers"

In [28]:
# Input window list
windows <- read.table(window_list_fn, header=F)
colnames(windows) <- c('Chr', 'Start', 'End')
LD_thresh = 0.1
# dim(windows)
# head(windows)

# read the model fit and span values for intervals after modelling LD-decay
decay_fit <- NULL
missing_files <- NULL
missing_iterations <- NULL
for (i in 1:nrow(windows)) {
    filename <- paste0(wdir, '/results/', windows$Chr[i], '_', windows$Start[i], '_', windows$End[i], '.', LD_thresh, '.txt')
    if (file.exists(filename)) {
        tmp <- read.table(filename, header=T)
        decay_fit <- rbind(decay_fit, cbind(Chr=windows$Chr[i], Start=windows$Start[i], End=windows$End[i], tmp))
    } else {
        missing_files <- c(missing_files, filename)
        missing_iterations <- c(missing_iterations, i)
    }
}
decay_fit <- decay_fit %>% as.data.frame()
# stopifnot(nrow(decay_fit) == nrow(windows))

dim(decay_fit)
head(decay_fit)

[1] 183   6

,Chr,Start,End,distance.span,expected.ld,mod_fit
,<int>,<int>,<int>,<int>,<dbl>,<dbl>
1,1,0,5000000,23118,0.09963293,0.9976277
2,1,2500000,7500000,32262,0.10089089,0.9971691
3,1,5000000,10000000,83356,0.10012954,0.9880250
4,1,7500000,12500000,262919,0.09996971,0.8192477
5,1,10000000,15000000,200722,0.09992422,0.6047200
6,1,12500000,17500000,218702,0.09988898,0.1849004


In [29]:
# Are there any missing files for the genomic interval LD estimation?
missing_iterations %>% length()
missing_iterations
# At this point check if there are missing LD estimates for genomic windows. Possible reason could be that script-3 did not run to completion

[1] 1

[1] 52

In [30]:
# assigning window spans for each of the SNPs based on overlap with genomic windows
unmatched_idx <- NULL
gwas_with_span <- NULL
for (i in 1:nrow(gwas_master_ss)) {
    tmp_decay_fit <- decay_fit %>% 
    filter(Chr==gwas_master_ss$Chr[i]) %>%
    mutate(snp_pos = gwas_master_ss$Pos[i]) %>%
    mutate(olap = ifelse(snp_pos >= Start & snp_pos <= End, 1, 0)) %>% 
    filter(olap == 1)
    # filter(Start <= gwas_master_ss$Pos[i] & End >= gwas_master_ss$Pos[i])

    if (nrow(tmp_decay_fit) == 0) {
        # SNP does not overlap with the window
        unmatched_idx = c(unmatched_idx, i)
        next
    } else if (nrow(tmp_decay_fit) == 1) {
        # SNP overlaps with one window
        span = tmp_decay_fit$distance.span[1]
        mod_fit = tmp_decay_fit$mod_fit[1]
    } else {
        # If SNP overlaps with more than one window taking the average
        span = round(mean(tmp_decay_fit$distance.span),0) 
        mod_fit = mean(tmp_decay_fit$mod_fit)
    }
    
    up_loc = gwas_master_ss$Pos[i] - span
    down_loc = gwas_master_ss$Pos[i] + span
    up_loc = ifelse(up_loc < 0, 0, up_loc)

    gwas_with_span <- rbind(gwas_with_span, cbind(gwas_master_ss[i,],
                                                  mod_fit=mod_fit,
                                                  span=span,
                                                  up_loc=up_loc,
                                                  down_loc=down_loc))
}

In [31]:
unmatched_idx %>% length()

[1] 0

In [32]:
dim(gwas_with_span)
stopifnot(nrow(gwas_master_ss) == nrow(gwas_with_span)) # checking if all SNPs have assigned LD spans
gwas_with_span %>% head()

[1] 2700   14

,SNP,Chr,Pos,P.value,MAF,nobs,H.B.P.Value,Effect,trait,model,mod_fit,span,up_loc,down_loc
,<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,2_68565136,2,68565136,1.081414e-10,0.003095975,323,2.136588e-04,0.26847596,tgw,BLINK,0.9793447,30848,68534288,68595984
2,2_68585251,2,68585251,1.615970e-08,0.222910217,323,7.981826e-03,0.02459134,tgw,BLINK,0.9793447,30848,68554403,68616099
3,3_5499477,3,5499477,4.449898e-10,0.055727554,323,4.395912e-04,0.04465348,tgw,BLINK,0.7830117,402410,5097067,5901887
4,3_27542756,3,27542756,5.228156e-09,0.034055728,323,3.443152e-03,-0.05415738,tgw,BLINK,0.4444400,420364,27122392,27963120
5,2_68585251,2,68585251,2.444744e-12,0.222910217,323,4.830170e-06,0.02720557,tgw,FarmCPU,0.9793447,30848,68554403,68616099
6,3_5499477,3,5499477,2.643453e-09,0.055727554,323,2.611383e-03,0.03446713,tgw,FarmCPU,0.7830117,402410,5097067,5901887


In [33]:
# Write the selected hits.
write.csv(gwas_with_span, file=paste0(wdir, '/fsnps/gwas_with_span.csv'), row.names=F, quote=F)

In [35]:
# write the list of unique coordinates to find genes associated with coordinates
query_intervals <- gwas_with_span %>% dplyr::select(Chr, up_loc, down_loc) %>% distinct() %>% arrange(Chr, up_loc)
write.table(query_intervals, file=paste0(wdir, '/fsnps/query_intervals.bed'), col.names=F, row.names=F, quote=F, sep="\t")

In [ ]:
# END